# Sequence weighting: GPU feasibility pilot

This experiment companion tests the pipeline needed to investigate Jane Street's
shared-versus-idiosyncratic-pattern hypothesis. It is **not an executed result**.
The local Ubuntu WSL 2 runtime has passed CUDA checks, and the equivalent
source scripts completed paired pilot runs. This notebook itself is unexecuted.

Run this notebook from the top in an approved CUDA GPU runtime. No external
dataset or API key is needed. The notebook embeds its research source files.

## Goal
Run one small causal Transformer for three epochs with random sequence weights,
then a paired uniform-weight control. Measure per-pattern learning, p*, timing,
and allocator memory. One seed and one capacity cannot establish a scaling law.

References: [Jane Street article](https://blog.janestreet.com/a-study-of-sequence-weighting-at-scale/)
and [estimator note](https://blog.janestreet.com/a-study-of-sequence-weighting-at-scale/sequence-weighting-note.pdf).

## Setup
Use a local Jupyter kernel with CUDA-capable PyTorch, NumPy, and Matplotlib.
The workspace provides a working Ubuntu WSL 2 Python environment; Jupyter
kernel setup is separate. The notebook never changes device security settings.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import subprocess
import sys

EXPERIMENT = Path('sequence-weighting-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S'))
CODE = EXPERIMENT / 'source'
CODE.mkdir(parents=True, exist_ok=False)
SEED = 42
TRAIN_SIZE = 2048
EPOCHS = 3
WIDTH = 128
LAYERS = 3
print('Experiment directory:', EXPERIMENT.resolve())


### Restore the embedded research source
The next cell writes the exact source files and verifies their hashes. It is
collapsed for readability; the generated `.py` files contain the inspectable
model, generator, estimator, and reporting code.


In [ ]:
SOURCES = {'core.py': '"""Dependency-free data generator and Jane Street exponent estimator."""\nimport math\nimport random\n\nVOCAB = 84\n\ndef exponent(weights, gains):\n    """Cumulative-discrepancy estimator, equivalent to d\' K d for unique ranks.\n\n    K_ij=min(r_i,r_j). Since sum(d)=0, cumulative prefix and suffix versions\n    coincide. Sorting avoids materializing an N by N matrix.\n    """\n    w = [float(x) for x in weights]\n    g = [float(x) for x in gains]\n    if max(w) - min(w) < 1e-12:\n        return {"p": None, "reason": "constant_weights"}\n    if sum(g) <= 1e-10:\n        return {"p": None, "reason": "nonpositive_total_gain"}\n    order = sorted(range(len(w)), key=w.__getitem__)\n    logw = [math.log(w[i]) for i in order]\n    total_gain = sum(g)\n    gn = [g[i] / total_gain for i in order]\n\n    def objective(p):\n        z = [p * x for x in logw]\n        m = max(z)\n        z = [math.exp(x - m) for x in z]\n        total = sum(z)\n        c = 0.0\n        loss = 0.0\n        for a, b in zip(gn, z):\n            c += a - b / total\n            loss += c * c\n        return loss / len(w)\n\n    grid = [i / 20 for i in range(161)]\n    j = min(range(len(grid)), key=lambda i: objective(grid[i]))\n    lo, hi = grid[max(0, j - 1)], grid[min(len(grid) - 1, j + 1)]\n    for _ in range(50):\n        a, b = lo + (hi - lo) / 3, hi - (hi - lo) / 3\n        if objective(a) < objective(b):\n            hi = b\n        else:\n            lo = a\n    candidates = [0.0, (lo + hi) / 2, 8.0]\n    p = min(candidates, key=objective)\n    return {"p": p, "objective": objective(p), "at_upper_bound": p >= 7.999,\n            "mean_gain": sum(g) / len(g)}\n\ndef make_data_lists(seed, regime, ntrain, nval, ntest):\n    rng = random.Random(seed)\n    permutations = [rng.sample(range(16), 16) for _ in range(16)]\n    ids = rng.sample(range(32 ** 3), ntrain + nval + ntest)\n    counts = {"mixed": [4, 4, 4], "structured": [6, 6, 0], "shared": [12, 0, 0]}[regime]\n    rows, labels = [], []\n    for i, key in enumerate(ids):\n        group = i % 16\n        row = [0, 1 + group, 17 + key // 1024, 17 + (key // 32) % 32, 17 + key % 32]\n        types = [t for t, count in enumerate(counts) for _ in range(count)]\n        rng.shuffle(types)\n        xs = rng.sample(range(16), 12)  # No repeated query keys within a sequence.\n        random_answers = [rng.randrange(16) for _ in xs]\n        mask = [-1] * 5\n        for typ, x, random_answer in zip(types, xs, random_answers):\n            y = (x + 1) % 16 if typ == 0 else permutations[group][x] if typ == 1 else random_answer\n            row += [49 + typ, 52 + x, 68 + y]\n            mask += [-1, -1, typ]\n        rows.append(row)\n        labels.append(mask[1:])  # Labels align with next-token targets.\n    bounds = [0, ntrain, ntrain + nval, len(rows)]\n    splits = {name: (rows[bounds[j]:bounds[j+1]], labels[bounds[j]:bounds[j+1]])\n              for j, name in enumerate(("train", "validation", "test"))}\n    meta = {"seed": seed, "regime": regime, "counts_per_sequence": counts,\n            "length": len(rows[0]), "vocab_size": VOCAB, "unique_keys": len(set(ids)),\n            "ntrain": ntrain, "nval": nval, "ntest": ntest,\n            "group_permutations": permutations, "sequence_keys": ids}\n    return splits, meta\n', 'core_checks.py': '"""Mathematical and dataset checks that do not import any training runtime."""\nimport json\nimport math\nfrom pathlib import Path\nfrom core import exponent, make_data_lists\n\n\ndef main():\n    weights = [math.exp(-3 + 6*i/99) for i in range(100)]\n    recovered = []\n    for expected in (0, .5, 1, 2):\n        result = exponent(weights, [2*w**expected for w in weights])\n        assert abs(result[\'p\']-expected) < 1e-5\n        recovered.append({\'known_p\': expected, \'estimated_p\': result[\'p\']})\n    gains = [1+.3*math.sin(i) for i in range(100)]\n    fit = exponent(weights, gains)\n    norm = sum(w**fit[\'p\'] for w in weights)\n    d = [g/sum(gains)-w**fit[\'p\']/norm for g,w in zip(gains,weights)]\n    direct = sum(d[i]*min((i+1)/100,(j+1)/100)*d[j] for i in range(100) for j in range(100))\n    assert abs(direct-fit[\'objective\']) < 1e-12\n    assert exponent([1,1], [1,2])[\'reason\'] == \'constant_weights\'\n    assert exponent([1,2], [-1,0])[\'reason\'] == \'nonpositive_total_gain\'\n    splits, meta = make_data_lists(1729, \'mixed\', 2048, 256, 512)\n    control, _ = make_data_lists(1729, \'shared\', 2048, 256, 512)\n    ids = meta[\'sequence_keys\']\n    assert len(ids) == len(set(ids)) == 2816\n    for name, (rows,kinds) in splits.items():\n        for row,mask,other in zip(rows,kinds,control[name][0]):\n            assert len(row) == 41 and len(mask) == 40\n            assert row[:5] == other[:5] and row[6::3] == other[6::3]\n            assert len(set(row[6::3])) == 12\n            assert all(mask.count(t) == 4 for t in range(3))\n            for index,typ in enumerate(mask):\n                if typ == 0:\n                    assert row[index+1]-68 == (row[index]-52+1)%16\n                elif typ == 1:\n                    assert row[index+1]-68 == meta[\'group_permutations\'][row[1]-1][row[index]-52]\n                elif typ == 2:\n                    assert 68 <= row[index+1] <= 83\n    status = {\n        \'status\': \'passed\', \'known_exponent_recovery\': recovered,\n        \'quadratic_form_equivalence_absolute_error\': abs(direct-fit[\'objective\']),\n        \'distinct_sequence_keys\': len(ids),\n        \'checks\': [\'constant weight and nonpositive gain handling\',\n                   \'all 2816 sequences: lengths, target alignment, pattern counts\',\n                   \'shared and group answer correctness\', \'unique inputs within each sequence\',\n                   \'disjoint train/validation/test keys\', \'cross-regime query pairing\'],\n        \'not_checked\': [\'Transformer causality at runtime\', \'CUDA training\', \'learning curves\',\n                        \'runtime performance or peak GPU memory\'],\n        \'training_status\': \'not performed by these dependency-free checks; inspect run outputs separately\'\n    }\n    output = Path(__file__).with_name(\'CORE_CHECKS.json\')\n    output.write_text(json.dumps(status, indent=2), encoding=\'utf-8\')\n    print(json.dumps(status, indent=2))\n\n\nif __name__ == \'__main__\':\n    main()\n', 'pilot.py': '"""Small, reproducible mechanism pilot inspired by Jane Street (2026).\n\nConditional next-token learning on synthetic sequences; not a replication of\nthe private benchmark. See README.md for scope and interpretation.\n"""\nimport argparse\nimport hashlib\nimport json\nimport math\nimport platform\nimport random\nimport time\nfrom pathlib import Path\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nTYPES = ("shared", "group", "instance")\nVOCAB = 84\n\n\nfrom core import exponent, make_data_lists\n\n\ndef make_data(seed, regime, ntrain, nval, ntest):\n    raw, metadata = make_data_lists(seed, regime, ntrain, nval, ntest)\n    return {name: tuple(torch.tensor(x, dtype=torch.long) for x in data)\n            for name, data in raw.items()}, metadata\n\n\nclass Block(nn.Module):\n    def __init__(self, width, heads):\n        super().__init__()\n        self.heads = heads\n        self.ln1, self.ln2 = nn.LayerNorm(width), nn.LayerNorm(width)\n        self.qkv = nn.Linear(width, 3 * width)\n        self.proj = nn.Linear(width, width)\n        self.ff = nn.Sequential(nn.Linear(width, 4 * width), nn.GELU(), nn.Linear(4 * width, width))\n\n    def forward(self, x):\n        b, t, c = x.shape\n        q, k, v = self.qkv(self.ln1(x)).reshape(b, t, 3, self.heads, c // self.heads).permute(2, 0, 3, 1, 4)\n        a = F.scaled_dot_product_attention(q, k, v, is_causal=True)\n        x = x + self.proj(a.transpose(1, 2).reshape(b, t, c))\n        return x + self.ff(self.ln2(x))\n\n\nclass Model(nn.Module):\n    def __init__(self, width, layers, length):\n        super().__init__()\n        self.token = nn.Embedding(VOCAB, width)\n        self.position = nn.Embedding(length, width)\n        self.blocks = nn.Sequential(*[Block(width, 4) for _ in range(layers)])\n        self.norm = nn.LayerNorm(width)\n        self.head = nn.Linear(width, VOCAB, bias=False)\n        self.apply(self.initialize)\n\n    @staticmethod\n    def initialize(module):\n        if isinstance(module, (nn.Linear, nn.Embedding)):\n            nn.init.normal_(module.weight, std=0.02)\n            if isinstance(module, nn.Linear) and module.bias is not None:\n                nn.init.zeros_(module.bias)\n\n    def forward(self, x):\n        h = self.token(x) + self.position(torch.arange(x.shape[1], device=x.device))[None]\n        return self.head(self.norm(self.blocks(h)))\n\n\ndef losses(model, tokens, kinds):\n    logits = model(tokens[:, :-1])\n    loss = F.cross_entropy(logits.transpose(1, 2), tokens[:, 1:], reduction="none")\n    valid = kinds >= 0\n    seq_loss = (loss * valid).sum(1) / valid.sum(1)\n    return seq_loss, loss, logits.argmax(-1).eq(tokens[:, 1:])\n\n\n@torch.no_grad()\ndef evaluate(model, data, device, batch):\n    model.eval()\n    seq, component, correct = [], {t: [] for t in TYPES}, {t: [] for t in TYPES}\n    for start in range(0, len(data[0]), batch):\n        tokens, kinds = (x[start:start+batch].to(device) for x in data)\n        sl, loss, acc = losses(model, tokens, kinds)\n        seq.append(sl.cpu())\n        for j, typ in enumerate(TYPES):\n            mask = kinds == j\n            if mask.any():\n                component[typ].append(loss[mask].cpu())\n                correct[typ].append(acc[mask].float().cpu())\n    out = {"loss": float(torch.cat(seq).mean())}\n    for typ in TYPES:\n        if component[typ]:\n            out[typ] = {"loss": float(torch.cat(component[typ]).mean()),\n                        "accuracy": float(torch.cat(correct[typ]).mean())}\n    return out, torch.cat(seq)\n\n\ndef validate():\n    weights = [math.exp(-3 + 6 * i / 99) for i in range(100)]\n    for expected in (0, .5, 1, 2):\n        fit = exponent(weights, [2 * w ** expected for w in weights])\n        assert abs(fit["p"] - expected) < 1e-5, (expected, fit)\n    # Independent direct quadratic-form check of the optimized estimator.\n    gains = [1 + .3 * math.sin(i) for i in range(100)]\n    fit = exponent(weights, gains)\n    p = fit["p"]\n    total = sum(w ** p for w in weights)\n    d = [g / sum(gains) - w ** p / total for g, w in zip(gains, weights)]\n    direct = sum(d[i] * min((i+1)/100, (j+1)/100) * d[j] for i in range(100) for j in range(100))\n    assert abs(direct - fit["objective"]) < 1e-12\n    assert exponent([1, 1], [1, 2])["p"] is None\n    assert exponent([1, 2], [-1, 0])["p"] is None\n    splits, meta = make_data(7, "mixed", 32, 16, 16)\n    control, _ = make_data(7, "shared", 32, 16, 16)\n    for name in splits:\n        assert torch.equal(splits[name][0][:, :5], control[name][0][:, :5])\n        assert torch.equal(splits[name][0][:, 6::3], control[name][0][:, 6::3])\n    assert meta["unique_keys"] == 64\n    for data in splits.values():\n        assert data[0].shape[1] == 41\n        assert ((data[1] >= 0).sum(1) == 12).all()\n        for row, mask in zip(*data):\n            for j in (mask == 0).nonzero().flatten().tolist():\n                assert int(row[j+1]) - 68 == (int(row[j]) - 52 + 1) % 16\n            for j in (mask == 1).nonzero().flatten().tolist():\n                assert int(row[j+1]) - 68 == meta["group_permutations"][int(row[1])-1][int(row[j])-52]\n    torch.manual_seed(9)\n    model = Model(32, 1, 40).eval()\n    x = splits["train"][0][:2, :-1].clone()\n    before = model(x).detach()\n    x[:, 20:] = (x[:, 20:] + 1) % VOCAB\n    assert torch.allclose(before[:, :20], model(x)[:, :20], atol=1e-6)\n    print("PASS: estimator recovery, quadratic equivalence, data alignment, disjoint keys, causal mask", flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--validate-only", action="store_true")\n    parser.add_argument("--output", type=Path, default=Path("work/runs/pilot"))\n    parser.add_argument("--regime", choices=("mixed", "structured", "shared"), default="mixed")\n    parser.add_argument("--weighting", choices=("random", "uniform"), default="random")\n    parser.add_argument("--width", type=int, default=128)\n    parser.add_argument("--layers", type=int, default=3)\n    parser.add_argument("--train-size", type=int, default=2048)\n    parser.add_argument("--epochs", type=int, default=3)\n    parser.add_argument("--batch", type=int, default=32)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--lr", type=float, default=0.0003)\n    args = parser.parse_args()\n    torch.set_num_threads(4)\n    if args.validate_only:\n        validate()\n        return\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA is required for this GPU benchmark; no silent CPU fallback.")\n    torch.manual_seed(args.seed)\n    torch.cuda.manual_seed_all(args.seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    args.output.mkdir(parents=True, exist_ok=False)\n    splits, metadata = make_data(1729, args.regime, args.train_size, 256, 512)\n    wrng = random.Random(1000 + args.seed)\n    raw = [math.exp(wrng.uniform(math.log(.01), math.log(10))) for _ in range(args.train_size)]\n    weights = torch.tensor(raw if args.weighting == "random" else [1.] * len(raw))\n    weights /= weights.mean()  # One corpus-level normalization, never batch normalization.\n    device = torch.device("cuda")\n    model = Model(args.width, args.layers, metadata["length"] - 1).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=.1)\n    config = {**vars(args), "output": str(args.output), "parameters": sum(p.numel() for p in model.parameters()),\n              "torch": torch.__version__, "python": platform.python_version(),\n              "gpu": torch.cuda.get_device_name(0), "precision": "float32", "weight_decay": .1,\n              "objective": "mean next-token NLL at 12 answer positions per sequence",\n              "pretrained": False, "loss_units": "nats/answer token",\n              "source_sha256": hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\n              "core_sha256": hashlib.sha256(Path(__file__).with_name("core.py").read_bytes()).hexdigest(),\n              "weight_min": float(weights.min()), "weight_max": float(weights.max()),\n              "weight_mean": float(weights.mean()),\n              "weight_effective_sample_size": float(weights.sum()**2 / weights.square().sum())}\n    (args.output / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")\n    (args.output / "data.json").write_text(json.dumps(metadata), encoding="utf-8")\n    torch.save(splits, args.output / "dataset.pt")\n    print(json.dumps(config), flush=True)\n    torch.cuda.reset_peak_memory_stats()\n    torch.cuda.synchronize()\n    start_time = time.perf_counter()\n    train_initial, initial_losses = evaluate(model, splits["train"], device, args.batch)\n    val_initial, _ = evaluate(model, splits["validation"], device, args.batch)\n    history = [{"epoch": 0, "train": train_initial, "validation": val_initial}]\n    sequence_losses = [initial_losses]\n    for epoch in range(1, args.epochs + 1):\n        model.train()\n        order = torch.randperm(args.train_size, generator=torch.Generator().manual_seed(999 + args.seed + epoch))\n        ep_start = time.perf_counter()\n        norms = []\n        for step, ids in enumerate(order.split(args.batch)):\n            tokens, kinds = (x[ids].to(device) for x in splits["train"])\n            sl, _, _ = losses(model, tokens, kinds)\n            loss = (sl * weights[ids].to(device)).mean()\n            optimizer.zero_grad(set_to_none=True)\n            loss.backward()\n            norm = nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            if not torch.isfinite(loss) or not torch.isfinite(norm):\n                raise RuntimeError("Non-finite loss or gradient; refusing to continue.")\n            norms.append(float(norm))\n            optimizer.step()\n            if (step + 1) % 32 == 0:\n                print(f"epoch={epoch} step={step+1} weighted_loss={float(loss):.4f}", flush=True)\n        torch.cuda.synchronize()\n        train_seconds = time.perf_counter() - ep_start\n        train_metrics, current = evaluate(model, splits["train"], device, args.batch)\n        val_metrics, _ = evaluate(model, splits["validation"], device, args.batch)\n        fitted = exponent(weights.tolist(), (initial_losses - current).tolist())\n        item = {"epoch": epoch, "train": train_metrics, "validation": val_metrics,\n                "p_star": fitted, "train_seconds": train_seconds,\n                "gradient_norm_mean": sum(norms) / len(norms),\n                "gradient_clip_fraction": sum(x > 1 for x in norms) / len(norms)}\n        history.append(item)\n        sequence_losses.append(current)\n        (args.output / "history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")\n        print(json.dumps(item), flush=True)\n    test, _ = evaluate(model, splits["test"], device, args.batch)\n    torch.cuda.synchronize()\n    result = {"config": config, "history": history, "final_test": test,\n              "elapsed_seconds": time.perf_counter() - start_time,\n              "peak_allocated_mib": torch.cuda.max_memory_allocated() / 2**20,\n              "peak_reserved_mib": torch.cuda.max_memory_reserved() / 2**20,\n              "warning": "Single-seed feasibility pilot; no scaling-law or statistical-significance claim."}\n    (args.output / "result.json").write_text(json.dumps(result, indent=2), encoding="utf-8")\n    torch.save({"weights": weights, "losses": torch.stack(sequence_losses)}, args.output / "sequence_losses.pt")\n    torch.save(model.state_dict(), args.output / "model.pt")\n    print(json.dumps({k: result[k] for k in ("elapsed_seconds", "peak_allocated_mib", "peak_reserved_mib", "final_test")}), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'report.py': '"""Build figures and a factual pilot report from completed run JSON files."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("runs", nargs="+", type=Path)\n    parser.add_argument("--output", type=Path, required=True)\n    args = parser.parse_args()\n    results = [json.loads((path / "result.json").read_text()) for path in args.runs]\n    max_epoch = max(r["config"]["epochs"] for r in results)\n    seeds = sorted({r["config"]["seed"] for r in results})\n    args.output.mkdir(parents=True, exist_ok=True)\n    plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10,\n                         "axes.spines.top": False, "axes.spines.right": False})\n    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3), layout="constrained")\n    colors = ["#156f9b", "#a85c19", "#3f7455", "#8555a4", "#333333"]\n    for i, result in enumerate(results):\n        cfg, history = result["config"], result["history"]\n        label = f\'{cfg["regime"]}, {cfg["weighting"]}, {cfg["parameters"]/1e6:.2f}M\'\n        available = [h for h in history if h.get("p_star", {}).get("p") is not None]\n        if available:\n            axes[0].plot([h["epoch"] for h in available],\n                         [h["p_star"]["p"] for h in available], marker="o",\n                         label=label, color=colors[i % len(colors)])\n        axes[1].plot([h["epoch"] for h in history], [h["validation"]["loss"] for h in history],\n                     marker="o", label=label, color=colors[i % len(colors)])\n    axes[0].set(title="Effective sequence-weight exponent", xlabel="Training epoch", ylabel="p* (dimensionless)")\n    axes[0].set_ylim(bottom=0)\n    axes[1].set(title="Validation answer loss", xlabel="Training epoch", ylabel="NLL (nats / answer token)")\n    for ax in axes:\n        ax.grid(axis="y", alpha=.2)\n        ax.legend(fontsize=8)\n        ax.set_xticks(range(max_epoch + 1))\n    fig.suptitle(f"Synthetic sequence-weighting pilot — exploratory, {len(seeds)} seed(s)", fontsize=13)\n    fig.savefig(args.output / "pilot-curves.png", dpi=180)\n    fig.savefig(args.output / "pilot-curves.svg")\n    plt.close(fig)\n\n    mixed = next((r for r in results if r["config"]["regime"] == "mixed" and r["config"]["weighting"] == "random"), None)\n    if mixed:\n        fig, ax = plt.subplots(figsize=(7.6, 4.4), layout="constrained")\n        for typ, color in zip(("shared", "group", "instance"), colors):\n            ax.plot([h["epoch"] for h in mixed["history"]],\n                    [h["validation"][typ]["accuracy"] * 100 for h in mixed["history"]],\n                    marker="o", color=color, label=typ)\n        ax.axhline(6.25, color="#666666", linestyle="--", label="16-answer chance: 6.25%")\n        ax.set(title="Mixed data, random weights: validation accuracy", xlabel="Training epoch", ylabel="Answer accuracy (%)", xticks=range(max_epoch + 1), ylim=(0, 105))\n        ax.legend()\n        ax.grid(axis="y", alpha=.2)\n        fig.savefig(args.output / "pattern-learning.png", dpi=180)\n        fig.savefig(args.output / "pattern-learning.svg")\n        plt.close(fig)\n\n    rows = []\n    for result in results:\n        cfg, final = result["config"], result["history"][-1]\n        rows.append({"regime": cfg["regime"], "weighting": cfg["weighting"], "parameters": cfg["parameters"],\n                     "seed": cfg["seed"], "final_p_star": final["p_star"]["p"],\n                     "p_star_details": final["p_star"], "epochs": cfg["epochs"], "train_size": cfg["train_size"],\n                     "initial_train_loss": result["history"][0]["train"]["loss"],\n                     "final_train_loss": final["train"]["loss"], "final_test": result["final_test"],\n                     "elapsed_seconds": result["elapsed_seconds"],\n                     "peak_allocated_mib": result["peak_allocated_mib"],\n                     "peak_reserved_mib": result["peak_reserved_mib"],\n                     "clipping_fractions": [h["gradient_clip_fraction"] for h in result["history"][1:]]})\n    (args.output / "summary.json").write_text(json.dumps(rows, indent=2), encoding="utf-8")\n    lines = ["# GPU pilot results", "", "These are measured exploratory results, not a scaling-law claim.", "",\n             "- Data: training count per run below; 256 validation, 512 test sequences; 12 answer tokens per sequence.",\n             "- Random initialization, float32, fixed final-epoch evaluation.",\n             f"- Seeds: {seeds}. This report does not compute between-seed confidence intervals.",\n             "- Memory figures describe the PyTorch allocator, excluding desktop/driver allocations.", ""]\n    lines += ["- Timings cover the instrumented training/evaluation section; interpreter imports, model/optimizer setup, package installation, and final checkpoint serialization are excluded.", ""]\n    for row in rows:\n        ptext = f\'undefined ({row["p_star_details"].get("reason")})\' if row["final_p_star"] is None else f\'{row["final_p_star"]:.5f}\'\n        lines += [f\'## {row["regime"]} / {row["weighting"]}\', "",\n                  f\'- Model parameters: {row["parameters"]:,}.\',\n                  f\'- Training sequences / epochs / seed: {row["train_size"]:,} / {row["epochs"]} / {row["seed"]}.\',\n                  f\'- Final p*: {ptext}.\',\n                  f\'- Training answer NLL: {row["initial_train_loss"]:.4f} -> {row["final_train_loss"]:.4f} nats/token.\',\n                  f\'- Final test answer NLL: {row["final_test"]["loss"]:.4f} nats/token.\',\n                  f\'- Elapsed training + evaluation: {row["elapsed_seconds"]:.1f} seconds.\',\n                  f\'- Peak allocated / reserved VRAM: {row["peak_allocated_mib"]:.1f} / {row["peak_reserved_mib"]:.1f} MiB.\',\n                  f\'- Clipped-step fractions by epoch: {row["clipping_fractions"]}.\']\n        for typ in ("shared", "group", "instance"):\n            if typ in row["final_test"]:\n                lines.append(f\'- {typ} test accuracy: {row["final_test"][typ]["accuracy"]*100:.2f}%.\')\n        lines.append("")\n    lines += ["## Interpretation limits", "",\n              "The original Jane Street study uses pretrained model families and an internal text benchmark. This pilot starts from random weights and applies sequence weights to conditional answer losses. The initial gain also includes learning that answer tokens occupy a restricted vocabulary. That shared gain can dilute p*. A low exponent alone therefore does not identify a capacity regime.", "",\n              "Uniform weights have no identifiable exponent. Rule accuracy on held-out examples is a diagnostic, not a replacement for the original training-set metric. One seed cannot establish a non-monotonic capacity curve or its epoch shift.", "",\n              "The next decision should depend on pattern learnability, gradient clipping, and fit quality. Freeze a validation-selected setup before running a multi-seed capacity ladder.", "",\n              "Source: https://blog.janestreet.com/a-study-of-sequence-weighting-at-scale/", ""]\n    (args.output / "RESULTS.md").write_text("\\n".join(lines), encoding="utf-8")\n    print(json.dumps(rows, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', 'README.md': "# Sequence weighting: local mechanism pilot\n\n**Current status:** the local Ubuntu WSL 2 CUDA runtime works, and both paired\nGPU feasibility runs completed. See `STATUS.md` and\n`results-local-wsl-20260928/RESULTS.md`. Smart App Control remains enabled;\nnative Windows PyTorch remains blocked. The portable notebook has not been\nexecuted end to end; the equivalent scripts were executed directly under WSL.\n\nThis executable pilot investigates the mechanism proposed in Jane Street's\n**A study of sequence weighting at scale** (14 September 2026).\n\n- Article: https://blog.janestreet.com/a-study-of-sequence-weighting-at-scale/\n- Estimator: https://blog.janestreet.com/a-study-of-sequence-weighting-at-scale/sequence-weighting-note.pdf\n\n## Scientific scope\n\nThe immediate objective is to verify that a causal Transformer, synthetic data,\nand the effective sequence-weight exponent can be measured on an RTX 3050 Laptop\nGPU with 4 GiB VRAM. Results from this pilot are exploratory. One model size and\none seed cannot establish a rise-then-fall curve or a scaling law.\n\nThe eventual paper question is whether learning shared versus idiosyncratic\npatterns explains both the non-monotonic exponent and its movement with epochs.\nThe public article does not release its private benchmark, so this is a\ncontrolled mechanism study, not an exact reproduction.\n\n## Dataset\n\nEach sequence contains a group identifier, a unique three-token example key,\nand 12 queries. Query inputs do not repeat within the sequence. Each query has\na mode, an input token, and an answer token:\n\n1. **Shared:** the answer is `(input + 1) mod 16` for every group and example.\n2. **Group:** the answer follows a fixed random permutation specific to the group.\n3. **Instance:** the answer is random and remains fixed when the training sequence\n   is revisited. New test examples have independent random answers.\n\nThere are 16 groups and 84 vocabulary tokens. Sequence length is 41 tokens.\nTraining, validation, and test use disjoint example keys. Groups and their rules\nare shared across splits; this explicitly tests within-group generalization,\nnot generalization to unseen groups. Chance accuracy for an answer conditional\non the 16-token answer vocabulary is 6.25%.\n\nThe mixed regime has four queries of each kind. Structured has six shared and\nsix group queries. Shared has 12 shared queries. Dataset randomness is fixed at\n1729 independently of training randomness. Example keys, groups, and query\ninputs are paired across regimes; query modes and their answers change as\nspecified by the experimental intervention.\n\n## Training and estimator\n\n- Causal Transformer trained from random initialization, float32, AdamW.\n- Mean next-token cross-entropy at the **answer positions only**, then a scalar\n  weight on each sequence. This is conditional sequence learning, not full-text\n  next-token pretraining. This is an explicit departure from a general LM study.\n- Fixed log-uniform sequence weights on `[0.01, 10]`, normalized once across the\n  training corpus to mean 1. We never normalize weights separately per batch.\n- Uniform-weight control uses the same initialization, examples, and per-epoch\n  shuffle for a given seed. Its exponent is undefined, not zero.\n- Three epochs; train and validation measured at each epoch; test measured only\n  at the fixed final epoch. No checkpoint selection using test performance.\n- Gains are measured relative to the **initial untrained model**, another\n  departure from the pretrained model families in Jane Street's study.\n- The estimator follows the normalized cumulative-discrepancy definition in\n  the note. It uses sorting and cumulative sums instead of an N by N kernel\n  matrix. Search range is `[0, 8]`; upper-bound hits are flagged. Nonpositive\n  aggregate gain is reported as undefined. Signed individual gains are retained.\n\n## Run on this workspace\n\nFrom the workspace root in PowerShell:\n\n```powershell\n& outputs\\sequence-weighting-pilot\\run-local.ps1\n```\n\nThis uses the existing Ubuntu WSL 2 distribution and `work/.venv-wsl`.\nIt checks CUDA, validates the pilot, runs the random/uniform pair, and builds\nthe report. The Windows environment `work/.venv` is retained for diagnostics;\nits CUDA import is blocked and it is not used by this launcher.\n\nEach output directory must be new to avoid overwriting a run. The program\nrequires working CUDA and will not silently substitute CPU training. Final\nconfigurations, epoch metrics, per-sequence loss arrays, weights, and model\nparameters are retained. Peak allocated and reserved memory describe PyTorch's\nallocator, not total system or desktop GPU memory.\n\n## Before a paper-scale experiment\n\n1. Inspect this pilot for correct target alignment, learnability, finite gains,\n   estimator behavior, timing, and available memory.\n2. Fix a small hyperparameter search using validation, then freeze it. This\n   pilot has not optimized regularization to held-out performance as the\n   original study did.\n3. Predefine a capacity ladder and data-complexity grid. Keep paired seeds and\n   weight assignments across capacities. Record exploratory changes separately.\n4. Use at least three training seeds, increasing repetitions when variance\n   requires it. Sequence bootstraps alone do not capture training variability.\n5. Add shared-rule pretraining before weighted adaptation to distinguish\n   random-initialization effects from adaptation of a pretrained model.\n6. Validate one setting on a public text dataset and an open pretrained model.\n\nIf no internal peak appears in the tested capacity range, report that range\nhonestly. It is neither evidence of a universal monotone law nor a refutation\nof the original large-scale observation.\n", 'protocol.md': "# Protocol v0.1 — 28 September 2026\n\n## Question and alignment\n\nCan controlled shared, group-specific, and instance-specific structure explain\nthe non-monotonic effective sequence-weight exponent and its movement with\ntraining duration reported by Jane Street?\n\nThe paper's central outcome is the training-set exponent. Generalization is a\nmechanistic diagnostic. We do not assume the original authors claimed that\ntheir exponent predicts downstream accuracy.\n\n## Stage 0: feasibility (authorized and implemented)\n\nUse 2,048 synthetic training sequences, 256 validation sequences, and 512 final\ntest sequences. One 128-wide, three-layer causal Transformer; batch size 32;\nAdamW, learning rate 0.0003, weight decay 0.1, gradient clipping at 1.0; three\nepochs. Start with mixed data, seed 42, random sequence weights and a paired\nuniform control. Report allocated/reserved VRAM, wall time, learning curves,\nexponent, component accuracy, and gradient-clipping frequency.\n\nThis stage is exploratory engineering validation. It is not a hypothesis test.\nNo hardware runtime estimates are asserted before measurement.\n\n## Stage 1: establish a usable task\n\nCheck that shared-rule accuracy improves beyond chance and that new-instance\nrandom-answer accuracy remains around 1/16. Inspect group-rule learnability.\nInspect the fitted cumulative curve as well as its scalar exponent. If a task\nis at floor or ceiling throughout the capacity ladder, change its complexity\nusing validation only and record the change as a new protocol version.\n\nInspect gradient clipping and the common loss reduction caused simply by\nlearning the answer-token vocabulary. An untrained baseline can dilute the\neffective exponent because all examples benefit from this shared improvement.\nCompare a shared-rule pretrained initialization before interpreting this as\nevidence about adaptation of pretrained language models.\n\n## Stage 2: mechanism study (not started)\n\nAfter feasibility, freeze a validation-selected learning setup and test a\ncapacity ladder in the same architecture family. Candidate configurations are\nwidth/layers: 64/2, 128/3, 256/4, 384/6. Actual parameter counts and feasibility\nmust be measured. Keep vocabulary, query count, data, and weight assignments\nconstant across capacities.\n\nData interventions: shared only; shared + group; shared + group + instance.\nEach model is evaluated after epochs 1, 2, 3, with a separately declared longer\ntraining arm if needed. Epoch checkpoints from one run are correlated; they\nare not independent experimental replicates.\n\nUse at least three independent training/weight seeds and report paired\ndifferences across capacities and interventions. Increase seed count based on\npilot variability, not on whether a preferred result becomes significant.\nKeep at least one independent data-generator seed for a robustness check.\n\n### Hypotheses\n\n- H1: Mixed shared/idiosyncratic structure can produce an interior maximum of\n  the exponent as capacity increases.\n- H2: Increased training duration can move that maximum toward smaller models.\n- H3: The movement coincides with changes in which components of the loss are\n  reduced; component learning explains more than parameter count alone.\n\nThese are conditional hypotheses, not guaranteed outcomes or universal laws.\n\n### Primary measurements\n\n- p-star and goodness-of-fit of its cumulative allocation curve.\n- Per-component training loss reduction from a fixed baseline checkpoint.\n- Location of the observed maximum on the tested capacity ladder, including\n  uncertainty across independent runs.\n\n### Secondary measurements\n\n- Validation/test rule accuracy and NLL; sequence-specific random answers\n  serve as a no-transfer control.\n- Gradient norm/clipping frequency; total answer tokens and processed tokens;\n  weight quantiles and effective sample size.\n- Sensitivity to learning rate, weight decay, and initialization, using a\n  predefined limited ablation rather than an unrestricted search.\n\n## Stage 3: external validation (not started)\n\nIf the mechanism is identifiable in the controlled study, adapt a small open\npretrained LM (initial candidate: Pythia 70M) on public text. If multiple model\nsizes do not fit or learn meaningfully, limit the claim to external validation\nat that size. LoRA is a separate adaptation-capacity intervention and must not\nbe silently substituted for full fine-tuning in a capacity comparison.\n\n## What would count as evidence?\n\n- Support: an interior peak and epoch movement reproduced across independent\n  runs, with the predicted pattern-component transition.\n- Boundary: an effect appears only in specified structure/optimization regimes.\n- Counterevidence to the proposed mechanism in our setting: the exponent\n  changes without the predicted component-learning transition.\n- Inconclusive: the explored ladder never reaches both sides of a peak,\n  learning remains at floor/ceiling, or between-run uncertainty is too large.\n\nA missing peak on a small synthetic ladder does not refute the original result\non very large pretrained models. A single-seed peak is not publishable evidence\nof a scaling law.\n\n## Sources\n\n- Jane Street article: https://blog.janestreet.com/a-study-of-sequence-weighting-at-scale/\n- Estimator note: https://blog.janestreet.com/a-study-of-sequence-weighting-at-scale/sequence-weighting-note.pdf\n- Byrd & Lipton, importance weighting: https://arxiv.org/abs/1812.03372\n- Pythia model suite: https://arxiv.org/abs/2304.01373\n- TinyStories, small-model experimentation: https://arxiv.org/abs/2305.07759\n"}
EXPECTED_SHA256 = {'core.py': '40867a7338b8eb63b97e593b4c48eed00e3d9719feac697b9c333f298fb312a3', 'core_checks.py': '39fd2cd3f10682fc293c977544ac3a80b8336b1a78a2764cc7e4723aa8d21abd', 'pilot.py': 'adc08f621ad7ddc7ffdc609b8305062d4e5f423f32c18f9b8ad03f7e024fb14c', 'report.py': '9f6b665dccb420ed382f64643e7e46dc024ac5981a4c0626e19f23115bd1e71a', 'README.md': '291daee922d2795200a6d392ef85049fa9d1191b2a56bd584d9ff952f42f9112', 'protocol.md': '46673b60c667875209e6ab7ca7347a39824ca6929432b15fac9349d9db02dc54'}
for name, source in SOURCES.items():
    assert hashlib.sha256(source.encode()).hexdigest() == EXPECTED_SHA256[name]
    (CODE / name).write_text(source, encoding='utf-8')
print('Source verified:', ', '.join(SOURCES))


## Checks
### Validate the mathematical estimator and synthetic data
These checks require only Python. They recover known exponents, compare the
cumulative estimator with the direct kernel quadratic form, and validate every
example's split key, target alignment, and shared/group rules.


In [ ]:
subprocess.run([sys.executable, str(CODE / 'core_checks.py')], check=True)


### Verify the CUDA runtime
This check must succeed before training. A missing GPU or blocked library stops
the notebook; it does not silently switch to CPU.


In [ ]:
import torch
import numpy
import matplotlib
assert torch.cuda.is_available(), 'Select an approved CUDA GPU runtime before training.'
print({'torch': torch.__version__, 'numpy': numpy.__version__,
       'matplotlib': matplotlib.__version__, 'gpu': torch.cuda.get_device_name(0),
       'vram_mib': torch.cuda.get_device_properties(0).total_memory / 2**20})
subprocess.run([sys.executable, str(CODE / 'pilot.py'), '--validate-only'], check=True)


## Steps
### Run the paired pilot
The mixed dataset has four shared-rule, four group-rule, and four instance-random
answers per sequence. It uses 2,048 training, 256 validation, and 512 final-test
examples, with disjoint example keys. Validation is logged at every epoch; test
is measured at the fixed final epoch. The model starts from random weights and
predicts answer tokens only, explicit departures from the original large-model study.

Both runs use the same initialization, data, and shuffle order. Uniform weights
have an **undefined** exponent. Do not interpret that as p*=0.


In [ ]:
RUNS = []
for weighting in ['random', 'uniform']:
    output = EXPERIMENT / ('mixed-' + weighting)
    command = [sys.executable, str(CODE / 'pilot.py'), '--output', str(output),
               '--weighting', weighting, '--regime', 'mixed', '--seed', str(SEED),
               '--train-size', str(TRAIN_SIZE), '--epochs', str(EPOCHS),
               '--width', str(WIDTH), '--layers', str(LAYERS)]
    subprocess.run(command, check=True)
    RUNS.append(output)


### Render the measured results
Plots and summary values are generated only from completed `result.json` files.
No expected or fabricated training results are embedded in this notebook.


In [ ]:
REPORT = EXPERIMENT / 'report'
subprocess.run([sys.executable, str(CODE / 'report.py'), *map(str, RUNS),
                '--output', str(REPORT)], check=True)
from IPython.display import Image, Markdown, display
display(Image(filename=str(REPORT / 'pilot-curves.png')))
display(Image(filename=str(REPORT / 'pattern-learning.png')))
display(Markdown((REPORT / 'RESULTS.md').read_text(encoding='utf-8')))


## Next steps
Inspect shared/group learnability, chance-level performance on new random
instance answers, clipping frequency, fit quality, and memory/time before
expanding the capacity ladder. An untrained baseline includes a shared gain
from recognizing the answer vocabulary; this can dilute p*. Add a shared-rule
pretraining control before making claims about pretrained-model adaptation.

Use independent seeds and a frozen validation-selected setup for a paper-scale
experiment. A single run cannot confirm a non-monotonic law or an epoch shift.


In [ ]:
import shutil
archive = shutil.make_archive(str(EXPERIMENT) + '-results', 'zip', root_dir=EXPERIMENT)
print('Saved reproducible experiment bundle:', archive)
